# Gold (Au) E-k dispersion from Materials Project

This notebook pulls the line-mode band structure for fcc gold (`mp-81`) and makes a first high-symmetry-path plot.

In [1]:
# Run once if needed:
# %pip install mp-api pymatgen ipywidgets


In [2]:
import os
from getpass import getpass
from mp_api.client import MPRester

API_KEY = os.getenv("MP_API_KEY")
if not API_KEY:
    API_KEY = getpass("Paste your Materials Project API key (input hidden): ").strip()
    if not API_KEY:
        raise RuntimeError("No API key provided.")
    os.environ["MP_API_KEY"] = API_KEY
    print("Using key entered in this notebook session.")
else:
    print("Using MP_API_KEY from environment.")

material_id = "mp-81"  # fcc Au

with MPRester(API_KEY) as mpr:
    try:
        bandstructure = mpr.get_bandstructure_by_material_id(material_id=material_id, line_mode=True)
    except TypeError:
        bandstructure = mpr.get_bandstructure_by_material_id(material_id)

print(f"Loaded {material_id}")
print(f"Fermi level: {bandstructure.efermi:.4f} eV")

Using key entered in this notebook session.


Retrieving ElectronicStructureDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Loaded mp-81
Fermi level: 5.8474 eV


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError as exc:
    raise ImportError(
        "ipywidgets is required for interactive controls. Install with: %pip install ipywidgets"
    ) from exc

# Optional: use local project style.
for candidate in (Path.cwd(), Path.cwd() / "code" / "main"):
    if (candidate / "plot_style.py").exists():
        sys.path.insert(0, str(candidate))
        try:
            from plot_style import apply_style
            apply_style()
        except Exception:
            pass
        break

spin_channel = list(bandstructure.bands.keys())[0]
energies = bandstructure.bands[spin_channel] - bandstructure.efermi  # shape: (nbands, nkpts)
distance = np.asarray(bandstructure.distance, dtype=float)
nbands, nkpts = energies.shape

# Collect labeled high-symmetry positions (supports repeated labels like X, L, etc.).
label_positions = {}
label_order = []
for i, kp in enumerate(bandstructure.kpoints):
    label = kp.label
    if not label:
        continue
    clean = label.replace(r"\Gamma", "Γ")
    x = float(distance[i])
    if clean not in label_positions:
        label_positions[clean] = []
        label_order.append(clean)
    if all(abs(x - x0) > 1e-10 for x0 in label_positions[clean]):
        label_positions[clean].append(x)

for lbl in label_positions:
    label_positions[lbl].sort()

if not label_positions:
    raise RuntimeError("No labeled k-points were found in this band structure.")

# Tick labels: merge multiple labels that fall at the same distance.
tick_label_map = {}
for lbl, xs in label_positions.items():
    for x in xs:
        if x in tick_label_map and lbl not in tick_label_map[x].split("|"):
            tick_label_map[x] = f"{tick_label_map[x]}|{lbl}"
        else:
            tick_label_map[x] = lbl

xmin = float(distance.min())
xmax = float(distance.max())
total_span = xmax - xmin

# Reasonable defaults: bands closest to EF and focus around X if available.
default_bands = tuple(int(i) for i in np.argsort(np.min(np.abs(energies), axis=1))[:4])
default_center = "X" if "X" in label_positions else ("L" if "L" in label_positions else label_order[0])
default_half_window = min(0.35, max(0.05, total_span / 4))
default_diff_a = int(default_bands[0]) if default_bands else 0
default_diff_b = int(default_bands[1]) if len(default_bands) > 1 else min(1, nbands - 1)
default_ea = energies[default_diff_a, :]
default_eb = energies[default_diff_b, :]
default_cross = ((default_ea < 0.0) & (default_eb > 0.0)) | ((default_ea > 0.0) & (default_eb < 0.0))
default_diff_curve = np.abs(default_ea - default_eb) * default_cross.astype(float)
default_hw_max = max(0.05, float(np.max(default_diff_curve)) * 1.05)
default_hw = min(default_hw_max, max(0.1, 0.4 * float(np.max(default_diff_curve))))

band_selector = widgets.SelectMultiple(
    options=[(f"band {i}", i) for i in range(nbands)],
    value=default_bands,
    description="Bands",
    rows=min(16, nbands),
    layout=widgets.Layout(width="260px", height="280px"),
)

view_mode = widgets.ToggleButtons(
    options=["Around point", "Full path"],
    value="Around point",
    description="View",
)

center_dropdown = widgets.Dropdown(
    options=label_order,
    value=default_center,
    description="Center point",
)

occurrence_dropdown = widgets.Dropdown(
    options=[],
    description="Occurrence",
)

half_window_slider = widgets.FloatSlider(
    value=default_half_window,
    min=0.02,
    max=max(0.05, total_span / 2),
    step=0.01,
    description="Δk/2 (1/Å)",
    readout_format=".2f",
    continuous_update=False,
)

shade_checkbox = widgets.Checkbox(
    value=True,
    description="Shade E < EF",
)

legend_checkbox = widgets.Checkbox(
    value=True,
    description="Show legend",
)

diff_band_a_dropdown = widgets.Dropdown(
    options=[(f"band {i}", i) for i in range(nbands)],
    value=default_diff_a,
    description="branch A",
)

diff_band_b_dropdown = widgets.Dropdown(
    options=[(f"band {i}", i) for i in range(nbands)],
    value=default_diff_b,
    description="branch B",
)

hbar_omega_slider = widgets.FloatSlider(
    value=default_hw,
    min=0.0,
    max=default_hw_max,
    step=0.01,
    description="hbar*omega",
    readout_format=".2f",
    continuous_update=False,
)

shade_diff_region_checkbox = widgets.Checkbox(
    value=True,
    description="Shade |dE| <= hbarw",
)


def update_occurrence_options(*_):
    xs = label_positions[center_dropdown.value]
    occurrence_dropdown.options = [(f"#{i + 1} @ {x:.3f}", i) for i, x in enumerate(xs)]
    occurrence_dropdown.value = 0


def toggle_focus_controls(*_):
    focus = view_mode.value == "Around point"
    center_dropdown.disabled = not focus
    occurrence_dropdown.disabled = not focus
    half_window_slider.disabled = not focus


def update_hbar_slider_range(*_):
    b1 = int(diff_band_a_dropdown.value)
    b2 = int(diff_band_b_dropdown.value)
    ea = energies[b1, :]
    eb = energies[b2, :]
    cross = ((ea < 0.0) & (eb > 0.0)) | ((ea > 0.0) & (eb < 0.0))
    diff_curve = np.abs(ea - eb) * cross.astype(float)
    local_max = float(np.max(diff_curve))
    new_max = max(0.05, 1.05 * local_max)
    hbar_omega_slider.max = new_max
    if hbar_omega_slider.value > new_max:
        hbar_omega_slider.value = new_max


center_dropdown.observe(update_occurrence_options, names="value")
view_mode.observe(toggle_focus_controls, names="value")
diff_band_a_dropdown.observe(update_hbar_slider_range, names="value")
diff_band_b_dropdown.observe(update_hbar_slider_range, names="value")
update_occurrence_options()
toggle_focus_controls()
update_hbar_slider_range()


def draw_plot(
    band_indices,
    mode,
    center_label,
    occurrence,
    half_window,
    shade_below_ef,
    show_legend,
    diff_band_a,
    diff_band_b,
    hbar_omega,
    shade_diff_region,
):
    band_indices = sorted({int(i) for i in band_indices})
    diff_band_a = int(diff_band_a)
    diff_band_b = int(diff_band_b)
    hbar_omega = float(hbar_omega)

    fig, (ax_band, ax_diff) = plt.subplots(
        2,
        1,
        figsize=(8.4, 7.0),
        sharex=True,
        gridspec_kw={"height_ratios": [3.3, 1.7], "hspace": 0.08},
    )

    if mode == "Around point":
        xs = label_positions[center_label]
        occ = min(max(int(occurrence), 0), len(xs) - 1)
        x0 = float(xs[occ])
        xlo = max(xmin, x0 - float(half_window))
        xhi = min(xmax, x0 + float(half_window))
    else:
        x0 = None
        xlo, xhi = xmin, xmax

    mask = (distance >= xlo - 1e-12) & (distance <= xhi + 1e-12)
    if not np.any(mask):
        mask = np.ones_like(distance, dtype=bool)

    # Upper panel: selected bands.
    if not band_indices:
        ax_band.text(0.5, 0.5, "Select at least one band.", ha="center", va="center", transform=ax_band.transAxes)
        ax_band.set_axis_off()
    else:
        for j, b in enumerate(band_indices):
            color = plt.get_cmap("tab10")(j % 10)
            first_segment = True
            for branch in bandstructure.branches:
                i0 = int(branch["start_index"])
                i1 = int(branch["end_index"]) + 1
                ax_band.plot(
                    distance[i0:i1],
                    energies[b, i0:i1],
                    color=color,
                    lw=1.35,
                    label=f"band {b}" if first_segment else None,
                )
                first_segment = False

        selected_energy = energies[band_indices, :]
        selected_energy = selected_energy[:, mask]
        emin = float(np.min(selected_energy))
        emax = float(np.max(selected_energy))
        pad = max(0.2, 0.08 * (emax - emin if emax > emin else 1.0))
        ax_band.set_ylim(emin - pad, emax + pad)

        ymin, ymax = ax_band.get_ylim()
        if shade_below_ef and ymin < 0.0:
            ax_band.axhspan(ymin, 0.0, color="0.88", alpha=0.3, zorder=-2)

        if show_legend and len(band_indices) <= 12:
            ax_band.legend(frameon=False, ncol=2, fontsize=9, loc="best")

    # Shared high-symmetry ticks and guide lines.
    local_tick_map = dict(tick_label_map)
    tick_x = [x for x in sorted(local_tick_map) if (x >= xlo - 1e-12 and x <= xhi + 1e-12)]
    if mode == "Around point" and x0 is not None and all(abs(x - x0) > 1e-10 for x in tick_x):
        tick_x.append(x0)
        tick_x.sort()
        local_tick_map[x0] = center_label

    for x in tick_x:
        ax_band.axvline(x, color="0.85", lw=0.8, zorder=0)
        ax_diff.axvline(x, color="0.85", lw=0.8, zorder=0)

    ax_band.set_xlim(xlo, xhi)
    ax_band.axhline(0.0, color="black", lw=0.95, ls="--", alpha=0.8)
    ax_band.set_title(f"Au band structure from Materials Project ({material_id})")
    ax_band.set_ylabel(r"$E - E_F$ (eV)")
    ax_band.tick_params(axis="x", labelbottom=False)

    # Lower panel: |dE| multiplied by 1 only when bands are on opposite sides of EF.
    ea = energies[diff_band_a, :]
    eb = energies[diff_band_b, :]
    cross_factor = ((ea < 0.0) & (eb > 0.0)) | ((ea > 0.0) & (eb < 0.0))
    diff_curve = np.abs(ea - eb) * cross_factor.astype(float)
    x_vis = distance[mask]
    diff_vis = diff_curve[mask]
    cross_vis = cross_factor[mask]

    ax_diff.plot(
        x_vis,
        diff_vis,
        color="tab:red",
        lw=1.5,
        label=f"|band {diff_band_a} - band {diff_band_b}| * I_opposite_side",
    )
    ax_diff.fill_between(x_vis, 0.0, diff_vis, color="0.92", alpha=0.8, zorder=-3)

    if shade_diff_region:
        region_mask = cross_vis & (diff_vis <= hbar_omega)
        ax_diff.fill_between(
            x_vis,
            0.0,
            diff_vis,
            where=region_mask,
            interpolate=True,
            color="tab:orange",
            alpha=0.45,
            label="shaded: |dE| <= hbarw",
            zorder=-2,
        )

    ax_diff.axhline(hbar_omega, color="black", lw=0.95, ls=":", alpha=0.85, label=f"hbar*omega = {hbar_omega:.2f} eV")
    ytop = max(float(np.max(diff_vis)), hbar_omega)
    ypad = max(0.05, 0.12 * (ytop if ytop > 0 else 1.0))
    ax_diff.set_ylim(0.0, ytop + ypad)
    ax_diff.set_ylabel(r"$|\Delta E|$ (eV)")
    ax_diff.set_xlabel("High-symmetry k-path")

    ax_diff.set_xticks(tick_x)
    ax_diff.set_xticklabels([local_tick_map[x] for x in tick_x])

    if show_legend:
        ax_diff.legend(frameon=False, fontsize=8, loc="upper right")

    plt.show()


ui = widgets.HBox(
    [
        widgets.VBox([band_selector]),
        widgets.VBox([view_mode, center_dropdown, occurrence_dropdown, half_window_slider, shade_checkbox, legend_checkbox]),
        widgets.VBox([diff_band_a_dropdown, diff_band_b_dropdown, hbar_omega_slider, shade_diff_region_checkbox]),
    ]
)

out = widgets.interactive_output(
    draw_plot,
    {
        "band_indices": band_selector,
        "mode": view_mode,
        "center_label": center_dropdown,
        "occurrence": occurrence_dropdown,
        "half_window": half_window_slider,
        "shade_below_ef": shade_checkbox,
        "show_legend": legend_checkbox,
        "diff_band_a": diff_band_a_dropdown,
        "diff_band_b": diff_band_b_dropdown,
        "hbar_omega": hbar_omega_slider,
        "shade_diff_region": shade_diff_region_checkbox,
    },
)

display(ui, out)


Output()

## Next build step

After this baseline plot, we can add projected bands (orbital character), export the k-point/energy arrays, and isolate X/L neighborhoods for direct comparison to your Rosei-like model.